# Floating Point Basics

This notebook is a first look at how computers store numbers.

Goals:
- see why `0.1 + 0.2` is not exactly `0.3`
- inspect FP32 bit patterns
- compare FP32, FP16, and an INT8-style approximation


In [ ]:
import struct
import numpy as np
import pandas as pd

result = 0.1 + 0.2
print("0.1 + 0.2 =", result)
print("Is it exactly 0.3?", result == 0.3)

for value in [0.1, 0.2, 0.3]:
    bits = struct.unpack(">I", struct.pack(">f", value))[0]
    print(f"{value:>3}: {bits:032b}")


In [ ]:
weights = np.array([0.12, -1.78, 3.14159, 0.004, -0.56], dtype=np.float32)
fp16 = weights.astype(np.float16)

# A simple fake INT8-style approximation using a scale factor.
scale = 32
int8_values = np.clip(np.round(weights * scale), -128, 127).astype(np.int8)
recovered = int8_values.astype(np.float32) / scale

comparison = pd.DataFrame(
    {
        "fp32": weights,
        "fp16": fp16.astype(np.float32),
        "fake_int8": recovered,
    }
)
comparison["fp16_error"] = (comparison["fp32"] - comparison["fp16"]).abs()
comparison["fake_int8_error"] = (comparison["fp32"] - comparison["fake_int8"]).abs()
comparison
